# "THE PRICE IS RIGHT" Capstone Project - Preprocessing Data

Building a model that predicts how much something costs from a description, based on a scrape of Amazon data

A model that can estimate how much something costs, from its description.

### Order of play

DAY 1: Data Curation

DAY 2: Data Pre-processing

DAY 3: Evaluation, Baselines, Traditional ML

DAY 4: Deep Learning and LLMs

DAY 5: Fine-tuning a Frontier Model

### DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.
LLMs are great at this!

<span style="color:green;">

### Business value of Data Curation

LLMs have made it simple to do something that was considered impossible only a few years ago. This approach can be applied to almost any business vertical, and it's similar to the advanced techniques RAG

</span>

In [19]:
# Importing Libraries

from litellm import completion
from huggingface_hub import login
from tqdm.notebook import tqdm
import os
import time
import json
from pricer.items import Item

In [2]:
# Log in to HuggingFace

hf_token = os.environ['HUGGING_FACE_WRITE_TOKEN']
login(hf_token, add_to_git_credential=True)

#### Dataset Selection

Use `LITE_MODE` = True for the free, fast version with training data size of 20,000

Use `LITE_MODE` =  False for the powerful, full version with training data size of 800,000

Please use full dataset if you have more local resource of high GPU or CPU else please go with smaller dataset

In [3]:
LITE_MODE = False

In [4]:
username = "Arivukkarasu"
dataset = f"{username}/Amazon_items_raw_lite" if LITE_MODE else f"{username}/Amazon_items_raw_full"
print(dataset)

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Arivukkarasu/Amazon_items_raw_full
Loaded 820,000 items
title='Did 520ATV298FB 520 ATV2 X-Ring Chain - 98 Links - Gold' category='Automotive' price=85.95 full='Did  520 ATV2 X-Ring Chain - 98 Links - Gold\n[\'Chain Application: OffroadChain Length: 98Chain Type: 520Marketing Color: GoldGreatly increased sealing performance with four sections. Keeps dirt out and lubrication in. Lowest friction of all types of O-rings. Twisting action disperses the pressure and minimizes power loss. Maximum wear resistance. Comes with clip-type connecting link.\']\n[\'Greatly increased sealing performance with four sections\', \'Keeps dirt out and lubrication in\', \'Lowest friction of all types of O-rings\', \'Twisting action disperses the pressure and minimizes power loss\', \'Maximum wear resistance\']\n{"Product Dimensions": "9 x 6 x 1 inches", "Item Weight": "3.49 pounds", "Manufacturer": "D.I.D", "Is Discontinued By Manufacturer": "No", "Date First Available": "November 25, 2018"}' weight=3.49 summ

In [5]:
items[0]

<Did 520ATV298FB 520 ATV2 X-Ring Chain - 98 Links - Gold = $85.95>

In [6]:
items[2].id

In [7]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [8]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [9]:
print(items[0].full)

Did  520 ATV2 X-Ring Chain - 98 Links - Gold
['Chain Application: OffroadChain Length: 98Chain Type: 520Marketing Color: GoldGreatly increased sealing performance with four sections. Keeps dirt out and lubrication in. Lowest friction of all types of O-rings. Twisting action disperses the pressure and minimizes power loss. Maximum wear resistance. Comes with clip-type connecting link.']
['Greatly increased sealing performance with four sections', 'Keeps dirt out and lubrication in', 'Lowest friction of all types of O-rings', 'Twisting action disperses the pressure and minimizes power loss', 'Maximum wear resistance']
{"Product Dimensions": "9 x 6 x 1 inches", "Item Weight": "3.49 pounds", "Manufacturer": "D.I.D", "Is Discontinued By Manufacturer": "No", "Date First Available": "November 25, 2018"}


In [12]:
start_time = time.time()
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/gemma3", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")
print(time.time() - start_time)

Title: D.I.D 520 ATV X-Ring Chain
Category: Automotive & Truck Parts
Brand: D.I.D
Description: This heavy-duty X-Ring chain provides superior performance and durability for off-road ATV applications.
Details: Featuring four sealing sections, this chain maximizes sealing performance while minimizing friction and wear.

Input tokens: 277
Output tokens: 74
Cost: 0.000 cents
16.477993726730347


In [17]:
MODEL = "ollama/gemma3"

In [21]:
def process_message(items):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items.full}]
    response = completion(messages=messages, model=MODEL, api_base="http://localhost:11434")
    return response.choices[0].message.content

Run the below cell if you have resource else we can download it from hugging face

In [ ]:
for i in len(items):
    summary = process_message(items[i])
    items[i].summary = summary

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

#### Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "Arivukkarasu"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)